In [ ]:
from datasets import load_dataset
from transformers import ViTFeatureExtractor
from modeling_vit import ViTForImageClassification
from transformers import TrainingArguments, Trainer
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader
from peft import LoraConfig, get_peft_model
from copy import deepcopy
from progbar import Progbar

/data/ai22mtech12002/anaconda3/envs/memcl/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [ ]:
adapters = torch.load('train_domain_adapters.pt')
feature_extractor = ViTFeatureExtractor.from_pretrained('google/vit-base-patch16-224-in21k')

@torch.no_grad()
def get_store_dict(model):
    module_weight_dict = {}
    for i in range(12):
        module = model.vit.encoder.layer[i]
        lorank_values = []
        for n, p in module.named_parameters():
            if "lora" in n:
                lorank_values.append(p.reshape(-1).detach())
        lorank_values = torch.cat(lorank_values, dim=0)
        module_weight_dict[i] = lorank_values
    module_weight_dict['classifier_weight'] = model.classifier.weight.reshape(-1).detach()
    module_weight_dict['classifier_bias'] = model.classifier.bias.reshape(-1).detach()
    return module_weight_dict

@torch.no_grad()
def set_store_dict(model, weight_dict):
    weight_dict = deepcopy(weight_dict)
    for i in range(12):
        module = model.vit.encoder.layer[i]
        for n, p in module.named_parameters():
            if "lora" in n:
                p.data = weight_dict[i][:p.numel()].reshape(p.shape)
                weight_dict[i] = weight_dict[i][p.numel():]
    model.classifier.weight.data = weight_dict['classifier_weight'].reshape(model.classifier.weight.shape)
    model.classifier.bias.data = weight_dict['classifier_bias'].reshape(model.classifier.bias.shape)

/data/ai22mtech12002/anaconda3/envs/memcl/lib/python3.12/site-packages/transformers/models/vit/feature_extraction_vit.py:28: FutureWarning: The class ViTFeatureExtractor is deprecated and will be removed in version 5 of Transformers. Please use ViTImageProcessor instead.
  warnings.warn(


In [3]:
# Load PACS dataset from Hugging Face
dataset = load_dataset("flwrlabs/pacs")

# Example access: domains are 'art_painting', 'cartoon', 'photo', 'sketch'
train_domains = ['art_painting', 'cartoon', 'photo', 'sketch']
train_dataset = dataset.filter(lambda x: x['domain'] in train_domains)['train']

In [4]:
model = ViTForImageClassification.from_pretrained('google/vit-base-patch16-224-in21k', num_labels=7).cuda()
model.eval()

Some weights of ViTForImageClassification were not initialized from the model checkpoint at google/vit-base-patch16-224-in21k and are newly initialized: ['classifier.bias', 'classifier.weight', 'encoder.layer.0.attention.attention.query_lora_A_weight', 'encoder.layer.0.attention.attention.query_lora_B_weight', 'encoder.layer.0.attention.attention.value_lora_A_weight', 'encoder.layer.0.attention.attention.value_lora_B_weight', 'encoder.layer.1.attention.attention.query_lora_A_weight', 'encoder.layer.1.attention.attention.query_lora_B_weight', 'encoder.layer.1.attention.attention.value_lora_A_weight', 'encoder.layer.1.attention.attention.value_lora_B_weight', 'encoder.layer.10.attention.attention.query_lora_A_weight', 'encoder.layer.10.attention.attention.query_lora_B_weight', 'encoder.layer.10.attention.attention.value_lora_A_weight', 'encoder.layer.10.attention.attention.value_lora_B_weight', 'encoder.layer.11.attention.attention.query_lora_A_weight', 'encoder.layer.11.attention.attent

ViTForImageClassification(
  (vit): ViTModel(
    (embeddings): ViTEmbeddings(
      (patch_embeddings): ViTPatchEmbeddings(
        (projection): Conv2d(3, 768, kernel_size=(16, 16), stride=(16, 16))
      )
      (dropout): Dropout(p=0.0, inplace=False)
    )
    (encoder): ViTEncoder(
      (layer): ModuleList(
        (0-11): 12 x ViTLayer(
          (attention): ViTSdpaAttention(
            (attention): ViTSdpaSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.0, inplace=False)
            )
            (output): ViTSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.0, inplace=False)
            )
          )
          (intermediate): ViTIntermediate(
            (dense): Linear(in_fe

In [ ]:
criterion = nn.CrossEntropyLoss()

for did, domain_name in enumerate(train_domains):
    set_store_dict(model, adapters[domain_name])
    train_dataset = dataset.filter(lambda x: x['domain'] == domain_name)['train']
    train_dataset = train_dataset.map(lambda x: feature_extractor(x['image']), batched=True)
    train_dataset.set_format(type='torch', columns=['pixel_values', 'label'])
    train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)

    pbar = Progbar(len(train_loader))
    for step, batch in enumerate(train_loader):
        pixel_values = batch['pixel_values'].cuda()
        labels = batch['label'].cuda()
        outputs = model(pixel_values)
        loss = criterion(outputs.logits, labels)

        acc = (outputs.logits.argmax(dim=1) == labels).float().mean().item()
        pbar.update(step + 1, values=[("loss", loss.item()), ("acc", acc)])

62/62 [==============================] - 46s 742ms/step - loss: 1.9090 - acc: 0.2493
